1. 문서내용읽기
2. 문서쪼개기
3. 임베딩 -> 백터 데이터베이스에 저장
4. 질문이 있을때, 백터 데이터베이스에 유사도 검색
5. 유사도 검색으로 가져온 문서를 LLM에 질문과 같이 전달

In [2]:
#문서쪼개기
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)

loader = Docx2txtLoader('./tax.docx')

document_list = loader.load_and_split(text_splitter=text_splitter)

In [9]:
#임베딩
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings

load_dotenv()

embedding = UpstageEmbeddings(model='solar-embedding-1-large')

In [12]:
from langchain_chroma import Chroma
#쪼개놓은 split_documents를 임베딩활용해서 database에 저장

# 데이터를 처음 저장할 때 
database = Chroma.from_documents(documents=document_list, embedding=embedding, collection_name='chroma-tax', persist_directory="./chroma")

# 이미 저장된 데이터를 사용할 때 
#database = Chroma(collection_name='chroma-tax', persist_directory="./chroma", embedding_function=embedding)

In [13]:
query = '연봉 5천만원인 직장인의 소득세는 얼마인가요?'
#retrieved_docs = database.similarity_search(query) #유사도 검사

In [32]:
retrieved_docs

[Document(id='3b5e5d0c-1e87-4c58-bfb6-60bf0d82e65f', metadata={'source': './tax.docx'}, page_content='2. 2명인 경우: 연 55만원\n\n3. 3명 이상인 경우: 연 55만원과 2명을 초과하는 1명당 연 40만원을 합한 금액\n\n② 삭제<2017. 12. 19.>\n\n③ 해당 과세기간에 출산하거나 입양 신고한 공제대상자녀가 있는 경우 다음 각 호의 구분에 따른 금액을 종합소득산출세액에서 공제한다.<신설 2015. 5. 13., 2016. 12. 20.>\n\n1. 출산하거나 입양 신고한 공제대상자녀가 첫째인 경우: 연 30만원\n\n2. 출산하거나 입양 신고한 공제대상자녀가 둘째인 경우: 연 50만원\n\n3. 출산하거나 입양 신고한 공제대상자녀가 셋째 이상인 경우: 연 70만원\n\n④ 제1항 및 제3항에 따른 공제를 “자녀세액공제”라 한다.<신설 2015. 5. 13., 2017. 12. 19.>\n\n[본조신설 2014. 1. 1.]\n\n[종전 제59조의2는 제59조의5로 이동 <2014. 1. 1.>]\n\n\n\n제59조의3(연금계좌세액공제) ① 종합소득이 있는 거주자가 연금계좌에 납입한 금액 중 다음 각 호에 해당하는 금액을 제외한 금액(이하 “연금계좌 납입액”이라 한다)의 100분의 12[해당 과세기간에 종합소득과세표준을 계산할 때 합산하는 종합소득금액이 4천 500만원 이하(근로소득만 있는 경우에는 총급여액 5천 500만원 이하)인 거주자에 대해서는 100분의 15]에 해당하는 금액을 해당 과세기간의 종합소득산출세액에서 공제한다. 다만, 연금계좌 중 연금저축계좌에 납입한 금액이 연 600만원을 초과하는 경우에는 그 초과하는 금액은 없는 것으로 하고, 연금저축계좌에 납입한 금액 중 600만원 이내의 금액과 퇴직연금계좌에 납입한 금액을 합한 금액이 연 900만원을 초과하는 경우에는 그 초과하는 금액은 없는 것으로 한다. <개정 2014. 12. 23., 2015.

문서를 벡터로 변환하는 임베딩 = solar-embedding-1-large
질문에 답변하는 대화형 AI = solar-mini4

In [14]:
from dotenv import load_dotenv
from langchain_upstage import ChatUpstage

load_dotenv()

llm = ChatUpstage(model="solar-mini4")

In [ ]:
#pull대신
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("human", """당신은 소득세 질문에 답변하는 도우미입니다.
검색된 문서를 참고하여 질문에 답변하세요.
문서에 답변의 근거가 없다면 모른다고 답변하세요.
답변은 간결하게 작성하세요.

Question: {question}
Context: {context}
Answer:""")
])

In [16]:
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm, retriever=database.as_retriever(), chain_type_kwargs={"prompt": prompt}
)

In [17]:
ai_message = qa_chain({"query": query})

C:\Users\u1ghd\AppData\Local\Temp\ipykernel_15268\3455095564.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  ai_message = qa_chain({"query": query})


In [18]:
ai_message

{'query': '연봉 5천만원인 직장인의 소득세는 얼마인가요?',
 'result': '연봉 5천만원인 직장인의 소득세는 **확정할 수 없습니다.**  \n계약서상 소득세는 연봉(총급여액)이 아니라 **근로소득공제·각종 소득공제·세액공제 후 산출된 종합소득과세표준**에 따라 달라지므로, 추가 정보(근로소득공제 적용 후 금액, 소득공제·세액공제 항목 등)가 필요합니다.\n\n다만, 주어진 세율표만으로 **종합소득과세표준이 5,000만원인 경우**의 산출세액은 다음과 같습니다.\n\n- 1,400만원 이하분: 1,400만원 × 6% = 84만원  \n- 1,400만원 초과 ~ 5,000만원 이하분: (5,000만원 − 1,400만원) × 15% = 3,600만원 × 15% = 540만원  \n- **종합소득산출세액 = 84만원 + 540만원 = 624만원**\n\n이는 세액공제·소득공제 적용 전 단계의 **산출세액**이며, 실제 납부할 소득세는 **근로소득세액공제 등 각종 세액공제를 반영한 후** 확정됩니다.'}